In [6]:
import os
import sys
import torch
import numpy as np
import pandas as pd
import json
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, DataCollatorWithPadding
from torch.utils.data import DataLoader


os.environ["TOKENIZERS_PARALLELISM"] = "false"

print(torch.__version__)



2.4.1+cu121


In [7]:
current_path = os.getcwd() 
data_path = os.path.join(current_path, "data")
books = os.path.join(current_path, "cleaned_books_data.json") 
print("books Path:", books)



books Path: /gpfs/home/mrios2/Desktop/Untitled Folder 1/cleaned_books_data.json


In [8]:
current_path = os.getcwd()
news_path = os.path.join(current_path, "data")
news = os.path.join(current_path, "cleaned_news_data.json") 
print("news Path:", news)



news Path: /gpfs/home/mrios2/Desktop/Untitled Folder 1/cleaned_news_data.json


In [9]:
acurrent_path = os.getcwd()
data_path = os.path.join(acurrent_path, "data")
books_path = os.path.join(data_path, "cleaned_books_data.json")

print("Books Path:", books_path)

with open(books_path, "r", encoding="utf-8") as f:
    books = json.load(f)

books_dataset = Dataset.from_list(books)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(example):
    return tokenizer(example["english"], truncation=True, padding="max_length", max_length=100)

tokenized_books = books_dataset.map(tokenize_function, batched=True)

split_dataset = tokenized_books.train_test_split(test_size=0.1)

books = DatasetDict({
    "train": split_dataset["train"],
    "test": split_dataset["test"]
})

books.set_format(type="torch", columns=["input_ids", "attention_mask"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

train_dataloader = DataLoader(
    books["train"],
    shuffle=True,
    batch_size=16,
    collate_fn=data_collator
)

test_dataloader = DataLoader(
    books["test"],
    shuffle=False,
    batch_size=16,
    collate_fn=data_collator
)

train_iterator = iter(train_dataloader)
sample_batch = next(train_iterator)

print("Sample batch keys:", sample_batch.keys())
print("Input IDs shape:", sample_batch["input_ids"].shape)


Books Path: /gpfs/home/mrios2/Desktop/Untitled Folder 1/data/cleaned_books_data.json


Map:   0%|          | 0/93470 [00:00<?, ? examples/s]

Sample batch keys: dict_keys(['input_ids', 'attention_mask'])
Input IDs shape: torch.Size([16, 100])


In [10]:
current_path = os.getcwd()
data_path = os.path.join(current_path, "data")
news_path = os.path.join(data_path, "cleaned_news_data.json")

print("News Path:", news_path)

with open(news_path, "r", encoding="utf-8") as f:
    news = json.load(f)

news_dataset = Dataset.from_list(news)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(example):
    return tokenizer(example["english"], truncation=True, padding="max_length", max_length=100)

tokenized_news = news_dataset.map(tokenize_function, batched=True)

split_dataset = tokenized_news.train_test_split(test_size=0.1)

news = DatasetDict({
    "train": split_dataset["train"],
    "test": split_dataset["test"]
})

news.set_format(type="torch", columns=["input_ids", "attention_mask"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

train_dataloader = DataLoader(
    news["train"],
    shuffle=True,
    batch_size=16,
    collate_fn=data_collator
)

test_dataloader = DataLoader(
    news["test"],
    shuffle=False,
    batch_size=16,
    collate_fn=data_collator
)

train_iterator = iter(train_dataloader)
sample_batch = next(train_iterator)

print("Sample batch keys:", sample_batch.keys())
print("Input IDs shape:", sample_batch["input_ids"].shape)


News Path: /gpfs/home/mrios2/Desktop/Untitled Folder 1/data/cleaned_news_data.json


Map:   0%|          | 0/112995 [00:00<?, ? examples/s]

Sample batch keys: dict_keys(['input_ids', 'attention_mask'])
Input IDs shape: torch.Size([16, 100])


In [11]:
full_dataset = torch.utils.data.ConcatDataset([books_dataset, news_dataset])
train_loader = DataLoader(full_dataset, batch_size=32, shuffle=True)


In [12]:
full_dataset_list = [dataset[i] for dataset in full_dataset.datasets for i in range(len(dataset))]

json_filename = "full_dataset.json"
with open(json_filename, "w", encoding="utf-8") as json_file:
    json.dump(full_dataset_list, json_file, ensure_ascii=False, indent=4)

print(f"Dataset saved as {json_filename}")
with open(json_filename, "r", encoding="utf-8") as f:
    full_data = json.load(f)  

print("First item in dataset:", full_data[0])


Dataset saved as full_dataset.json
First item in dataset: {'id': 'en/Austen_Jane-Sense_and_Sensibility.xml.gz\tes/Austen_Jane-Sense_and_Sensibility.xml.gz\ts1\ts1', 'english': 'Source: Project GutenbergAudiobook available here', 'spanish': 'Source: Wikisource & librodot.com', 'id_english': 'en/Austen_Jane-Sense_and_Sensibility.xml.gz', 'id_spanish': 'es/Austen_Jane-Sense_and_Sensibility.xml.gz', 'segment_id': 's1', 'alignment_id': 's1'}


In [13]:
current_path = os.getcwd()
data_path = os.path.join(current_path, "data")
full_path = os.path.join(data_path, "full_dataset.json")

print("Full Path:", full_path)

with open(full_path, "r", encoding="utf-8") as f:
    full_data = json.load(f)

full_dataset = Dataset.from_list(full_data)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(example):
    return tokenizer(example["english"], truncation=True, padding="max_length", max_length=100)

tokenized_dataset = full_dataset.map(tokenize_function, batched=True)

split_dataset = tokenized_dataset.train_test_split(test_size=0.1)

full_dataset_dict = DatasetDict({
    "train": split_dataset["train"],
    "test": split_dataset["test"]
})

full_dataset_dict.set_format(type="torch", columns=["input_ids", "attention_mask"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

train_dataloader = DataLoader(
    full_dataset_dict["train"],
    shuffle=True,
    batch_size=16,
    collate_fn=data_collator
)

test_dataloader = DataLoader(
    full_dataset_dict["test"],
    shuffle=False,
    batch_size=16,
    collate_fn=data_collator
)

train_iterator = iter(train_dataloader)
sample_batch = next(train_iterator)

print("Sample batch keys:", sample_batch.keys())
print("Input IDs shape:", sample_batch["input_ids"].shape)


Full Path: /gpfs/home/mrios2/Desktop/Untitled Folder 1/data/full_dataset.json


Map:   0%|          | 0/206465 [00:00<?, ? examples/s]

Sample batch keys: dict_keys(['input_ids', 'attention_mask'])
Input IDs shape: torch.Size([16, 100])


In [14]:

with open(full_path, "r", encoding="utf-8") as f:
    full_dataset = json.load(f)  


split_index = int(0.9 * len(full_dataset))
train_data = full_dataset[:split_index]
test_data = full_dataset[split_index:]
books_dataset = DatasetDict({
    "train": Dataset.from_list(train_data),
    "test": Dataset.from_list(test_data)
})

books_train_dataset = books_dataset["train"]
print(books_train_dataset.column_names)


['id', 'english', 'spanish', 'id_english', 'id_spanish', 'segment_id', 'alignment_id']


In [15]:
train_iterator = iter(train_dataloader)
sample_batch = next(train_iterator)  
print(sample_batch)


{'input_ids': tensor([[  101,  1996,  5679,  ...,     0,     0,     0],
        [  101, 29536,  2271,  ...,     0,     0,     0],
        [  101,  5629, 17152,  ...,     0,     0,     0],
        ...,
        [  101,  1000,  6289,  ...,     0,     0,     0],
        [  101,  2016,  2052,  ...,     0,     0,     0],
        [  101,  1999,  2300,  ...,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}


In [16]:
train_iterator = iter(train_dataloader)
sample_batch = next(train_iterator)  

print(sample_batch.keys())
print(sample_batch['input_ids'].shape)
print(sample_batch['input_ids'])


dict_keys(['input_ids', 'attention_mask'])
torch.Size([16, 100])
tensor([[  101,  1000,  1037,  ...,     0,     0,     0],
        [  101,  1996,  2622,  ...,     0,     0,     0],
        [  101,  1005,  1996,  ...,     0,     0,     0],
        ...,
        [  101,  1037, 25312,  ...,     0,     0,     0],
        [  101,  1996,  3618,  ...,     0,     0,     0],
        [  101,  1005,  2748,  ...,     0,     0,     0]])


In [17]:
full_dataset = Dataset.from_list(full_dataset_list)  # Convert list to Hugging Face Dataset


split_dataset = full_dataset.train_test_split(test_size=0.2, seed=42)
print(split_dataset) 

train_dataset = split_dataset["train"]
print(train_dataset.column_names)


DatasetDict({
    train: Dataset({
        features: ['id', 'english', 'spanish', 'id_english', 'id_spanish', 'segment_id', 'alignment_id'],
        num_rows: 165172
    })
    test: Dataset({
        features: ['id', 'english', 'spanish', 'id_english', 'id_spanish', 'segment_id', 'alignment_id'],
        num_rows: 41293
    })
})
['id', 'english', 'spanish', 'id_english', 'id_spanish', 'segment_id', 'alignment_id']


In [18]:
def add_labels(example):
    example["labels"] = int(example.get("alignment_id") is not None)  
    return example

split_dataset = split_dataset.map(add_labels)
print(split_dataset["train"].column_names)  




Map:   0%|          | 0/165172 [00:00<?, ? examples/s]

Map:   0%|          | 0/41293 [00:00<?, ? examples/s]

['id', 'english', 'spanish', 'id_english', 'id_spanish', 'segment_id', 'alignment_id', 'labels']


In [19]:
with open(full_path, "r", encoding="utf-8") as f:
    full_dataset = json.load(f)

full_dataset = Dataset.from_list(full_dataset)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(example):
    return tokenizer(example["english"], truncation=True, padding="max_length", max_length=100)

tokenized_dataset = full_dataset.map(tokenize_function, batched=True)

def add_labels(example):
    example["labels"] = int(example.get("alignment_id", 0) is not None)
    return example

if "labels" not in tokenized_dataset.column_names:
    tokenized_dataset = tokenized_dataset.map(add_labels)

tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(tokenized_dataset.column_names)


Map:   0%|          | 0/206465 [00:00<?, ? examples/s]

Map:   0%|          | 0/206465 [00:00<?, ? examples/s]

['id', 'english', 'spanish', 'id_english', 'id_spanish', 'segment_id', 'alignment_id', 'input_ids', 'attention_mask', 'labels']


In [20]:
train_dataloader = DataLoader(tokenized_dataset, batch_size=16, shuffle=True)

batch = next(iter(train_dataloader))
print(batch.keys()) 


dict_keys(['input_ids', 'attention_mask', 'labels'])


In [21]:
import torch.nn as nn

all_labels = torch.tensor(tokenized_dataset["labels"]) 
unique, counts = torch.unique(all_labels, return_counts=True)
class_counts = dict(zip(unique.tolist(), counts.tolist()))

class_0_count = class_counts.get(0, 1)
class_1_count = class_counts.get(1, 1)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([class_0_count / class_1_count]))

print(f"Class 0 count: {class_0_count}, Class 1 count: {class_1_count}")


/tmp/ipykernel_2612712/1303119413.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  all_labels = torch.tensor(tokenized_dataset["labels"])


NameError: name 'nn' is not defined